# 🎨 Image Synthesis from Semantic Labeling — Conditional GAN

This notebook implements a **Conditional GAN (cGAN)** for synthesizing photorealistic images from semantic label maps, following the **pix2pix** family of architectures.

---

## 🗺️ Overview

| Module | Role |
|--------|------|
| **Generator** (U-Net) | Takes a semantic label map → outputs a synthesized image |
| **Discriminator** (PatchGAN) | Decides if `[label_map, image]` pairs look real or fake |
| **Losses** | GANLoss + L1 Reconstruction + Feature Matching |

### 🔄 Training Loop (conceptual)
```
Label Map ──► Generator ──► Fake Image
                                 │
                    ┌────────────┘
                    ▼
Discriminator([label, fake]) ──► fake score
Discriminator([label, real]) ──► real score
                    │
                    ▼
           GAN Loss + L1 + Feature Matching
```

---
**Requirements:** `torch`, `torchvision`

## 📦 Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
# 🏗️ Part 1 — Generator (U-Net)

## What is a U-Net?

A U-Net has two paths:
- **Encoder** (left side): progressively *downsamples* the input, capturing high-level semantics
- **Decoder** (right side): progressively *upsamples* back to full resolution
- **Skip connections**: copy encoder feature maps directly to the decoder at each matching resolution

```
Input (256×256)
  │
  ▼ Encoder (×8 blocks, stride=2)
  │   E1 → 128×128
  │   E2 → 64×64
  │   ...   ...
  │   E8 → 1×1  ← bottleneck
  │
  ▼ Decoder (×7 blocks, stride=2 upsample)
  │   D1 + skip(E7) → 2×2
  │   D2 + skip(E6) → 4×4
  │   ...   ...
  │   D7 + skip(E1) → 128×128
  │
  ▼ Final ConvTranspose → 256×256
  │
 Tanh  →  output in [-1, 1]
```

Skip connections are crucial — they preserve fine spatial detail (edges, textures) that would otherwise be lost during downsampling.

### 1.1 Encoder Block

Each encoder step **halves** the spatial resolution and **doubles** the channel count.

```
x  ──►  Conv2d(4×4, stride=2)  ──►  [BatchNorm]  ──►  LeakyReLU(0.2)
```

- **`stride=2`**: halves H and W
- **`kernel=4, padding=1`**: standard pix2pix sizing
- **`LeakyReLU(0.2)`**: allows small gradients for negative values (avoids dead neurons)
- **No BN on first layer**: because the input is raw pixel data (normalization would hurt)

In [ ]:
class EncoderBlock(nn.Module):
    """
    Single downsampling block used in the U-Net encoder.

    Structure:
        Conv2d (stride=2) → [BatchNorm2d] → LeakyReLU(0.2)

    Args:
        in_channels:    Number of input feature channels.
        out_channels:   Number of output feature channels.
        use_batchnorm:  Whether to apply BatchNorm (False for the very first block).

    Output spatial size: H/2 × W/2
    """

    def __init__(self, in_channels, out_channels, use_batchnorm=True):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=4,   # Standard pix2pix kernel size
                stride=2,        # Halves spatial resolution
                padding=1,       # Keeps output exactly H/2 × W/2
                bias=not use_batchnorm  # BN has its own bias — no need for conv bias
            )
        ]

        if use_batchnorm:
            # Normalize activations across the batch: stabilizes training
            layers.append(nn.BatchNorm2d(out_channels))

        # LeakyReLU: unlike ReLU, allows a small gradient for negative values.
        # The slope 0.2 is standard in GAN discriminators and encoders.
        layers.append(nn.LeakyReLU(0.2, inplace=True))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

### 1.2 Decoder Block

Each decoder step **doubles** the spatial resolution and concatenates the skip connection from the matching encoder layer.

```
x  ──►  ConvTranspose2d(4×4, stride=2)  ──►  BatchNorm  ──►  [Dropout]  ──►  ReLU
                                                                                  │
                                                                cat([·, skip]) ◄──┘
```

- **`ConvTranspose2d`**: learnable upsampling ("deconvolution")
- **`Dropout(0.5)`**: applied to the first 3 decoder blocks only — adds noise during training for better generalization
- **`ReLU`**: decoder uses regular ReLU (vs. LeakyReLU in encoder)
- **Skip concat**: doubles the channel count after concatenation, giving the decoder access to fine-grained encoder features

In [ ]:
class DecoderBlock(nn.Module):
    """
    Single upsampling block used in the U-Net decoder.

    Structure:
        ConvTranspose2d (stride=2) → BatchNorm2d → [Dropout(0.5)] → ReLU
        then: cat([upsampled, skip_connection])  ← channel count doubles

    Args:
        in_channels:   Number of input channels (from previous decoder layer).
        out_channels:  Number of output channels (before skip concatenation).
        use_dropout:   If True, adds Dropout(0.5) — used in the first 3 decoder blocks.

    Note:
        After forward(), the output tensor has `out_channels + skip_channels` channels
        because of the skip connection concatenation.
    """

    def __init__(self, in_channels, out_channels, use_dropout=False):
        super().__init__()

        layers = [
            nn.ConvTranspose2d(
                in_channels, out_channels,
                kernel_size=4,
                stride=2,    # Doubles spatial resolution
                padding=1,
                bias=False   # BatchNorm follows, so bias is redundant
            ),
            nn.BatchNorm2d(out_channels),
        ]

        if use_dropout:
            # Dropout only in the deepest decoder layers (closest to bottleneck).
            # This injects noise during training, acting as a form of regularization.
            layers.append(nn.Dropout(0.5))

        # ReLU (not LeakyReLU) is standard for the decoder path
        layers.append(nn.ReLU(inplace=True))

        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        """
        Args:
            x:    Feature map from the previous decoder layer [B, C, H, W].
            skip: Corresponding encoder feature map [B, C_skip, H, W].

        Returns:
            Tensor of shape [B, C + C_skip, 2H, 2W]
        """
        x = self.block(x)                  # Upsample: H×W → 2H×2W
        return torch.cat([x, skip], dim=1)  # Concat along channel axis

### 1.3 Full U-Net Generator

The generator assembles encoder and decoder blocks into a symmetric U-Net.

**Channel progression** (default: `ngf=64`, `depth=8`, input 256×256):

| Layer | Resolution | Channels |
|-------|-----------|----------|
| Input | 256×256 | 3 |
| E1 | 128×128 | 64 |
| E2 | 64×64 | 128 |
| E3 | 32×32 | 256 |
| E4–E8 | 16×16 → 1×1 | 512 (capped) |
| D1–D7 | 2×2 → 128×128 | mirror + skip |
| Output | 256×256 | 3 |

In [ ]:
class UNetGenerator(nn.Module):
    """
    U-Net Generator for conditional image synthesis.

    Converts a semantic label map into a photorealistic image using a
    symmetric encoder-decoder architecture with skip connections.

    Args:
        input_nc:  Channels in the input label map (e.g. 3 for RGB labels).
        output_nc: Channels in the synthesized image (e.g. 3 for RGB).
        ngf:       Base filter count; doubles at each encoder level (capped at ngf*8).
        depth:     Number of encoder/decoder stages. Use 8 for 256×256 inputs.
        dropout:   Dropout rate applied to the first 3 decoder blocks.

    Forward pass:
        Input:  [B, input_nc,  H, W]
        Output: [B, output_nc, H, W]  — values in [-1, 1] (Tanh output)
    """

    def __init__(self, input_nc=3, output_nc=3, ngf=64, depth=8, dropout=0.5):
        super().__init__()
        assert depth >= 4, "U-Net depth must be >= 4"
        self.depth = depth

        # ── Encoder ──────────────────────────────────────────────────────
        # Builds `depth` downsampling blocks.
        # Channels: input_nc → ngf → ngf*2 → ngf*4 → ngf*8 → ngf*8 → ...
        # (channel count is capped at ngf*8 to limit memory usage)
        self.encoders = nn.ModuleList()
        encoder_channels = []  # Track output channels for building the decoder

        in_ch = input_nc
        for i in range(depth):
            if i == 0:
                out_ch = ngf
                use_bn = False   # First block: raw input, skip BatchNorm
            elif i <= 3:
                out_ch = ngf * min(2 ** i, 8)  # 128, 256, 512
                use_bn = True
            else:
                out_ch = ngf * 8  # Cap at 512 — prevents excessive memory use
                use_bn = True

            self.encoders.append(EncoderBlock(in_ch, out_ch, use_batchnorm=use_bn))
            encoder_channels.append(out_ch)  # Save for later skip-connection matching
            in_ch = out_ch

        # ── Decoder ──────────────────────────────────────────────────────
        # Mirror of the encoder. At each step:
        #   - Upsample from previous decoder output
        #   - Concatenate the matching encoder feature map (skip connection)
        self.decoders = nn.ModuleList()

        in_ch = encoder_channels[-1]  # Start from bottleneck output
        for i in range(depth - 1):
            enc_idx = depth - 2 - i   # Which encoder level to skip-connect with
            skip_ch = encoder_channels[enc_idx]

            use_drop = (i < 3) and (dropout > 0)  # Dropout only in early decoder blocks
            out_ch = skip_ch

            self.decoders.append(DecoderBlock(in_ch, out_ch, use_dropout=use_drop))

            # After cat([decoded, skip]): channels = out_ch + skip_ch
            in_ch = out_ch + skip_ch

        # ── Final output layer ────────────────────────────────────────────
        # Upsample to full resolution → apply Tanh to get values in [-1, 1]
        self.final = nn.Sequential(
            nn.ConvTranspose2d(in_ch, output_nc, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # Output range [-1, 1] to match normalized image targets
        )

        # Initialize all Conv and BN weights using the pix2pix convention
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        """
        Weight initialization from the original pix2pix paper:
          - Conv weights: Normal(mean=0, std=0.02)
          - BN weights:   Normal(mean=1, std=0.02),  bias=0

        This narrow initialization prevents saturating activations at the
        start of training.
        """
        classname = m.__class__.__name__
        if classname.find("Conv") != -1:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif classname.find("BatchNorm") != -1:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

    def forward(self, x):
        """
        Args:
            x: Semantic label map [B, input_nc, H, W]

        Returns:
            Synthesized image [B, output_nc, H, W] with values in [-1, 1]
        """
        # ── Encoder: save all intermediate feature maps for skip connections
        encoder_features = []
        h = x
        for encoder in self.encoders:
            h = encoder(h)
            encoder_features.append(h)  # Save before next downsampling

        # ── Decoder: upsample + merge skip connections in reverse order
        h = encoder_features[-1]  # Start from bottleneck (smallest feature map)
        for i, decoder in enumerate(self.decoders):
            skip = encoder_features[self.depth - 2 - i]  # Matching encoder level
            h = decoder(h, skip)  # Upsample + concatenate skip

        # ── Final layer: reach original resolution
        return self.final(h)

### ✅ Test the Generator

In [ ]:
# Instantiate the generator
gen = UNetGenerator(input_nc=3, output_nc=3, ngf=64, depth=8)

# Simulate a batch of 2 semantic label maps at 256×256
x = torch.randn(2, 3, 256, 256)
out = gen(x)

print(f"Input  shape: {x.shape}    (batch=2, channels=3, 256×256)")
print(f"Output shape: {out.shape}  (should match input spatial size)")
print(f"Output range: [{out.min():.2f}, {out.max():.2f}]  (Tanh → [-1, 1])")

total_params = sum(p.numel() for p in gen.parameters())
print(f"Total parameters: {total_params:,}")

assert out.shape == (2, 3, 256, 256), f"Unexpected shape: {out.shape}"
print("✓ Generator test passed")

---
# 🔍 Part 2 — PatchGAN Discriminator

## What is PatchGAN?

Instead of outputting a **single** real/fake score for the whole image, a PatchGAN outputs a **2D grid** of scores — one per overlapping patch of the input.

With `n_layers=3`, each output cell covers a **70×70 pixel receptive field** in the input.

```
[label_map | image]  →  Conv blocks  →  Spatial score map [B, 1, H', W']
   (6 channels)                           each cell = real/fake for a patch
```

**Why PatchGAN?**
- Focuses on local texture quality rather than global structure
- Fewer parameters than a full-image discriminator
- Effective at enforcing sharp, realistic textures

The discriminator is **conditional**: it sees both the label map and the image together, so it learns whether the generated image is consistent with the given labels.

### 2.1 Discriminator Block

Same conv structure as the encoder, but used inside the discriminator:
```
Conv2d(4×4) → [BatchNorm] → LeakyReLU(0.2)
```

In [ ]:
class DiscriminatorBlock(nn.Module):
    """
    A single convolutional block for the PatchGAN discriminator.

    Structure:
        Conv2d(4×4) → [BatchNorm2d] → LeakyReLU(0.2)

    Args:
        in_channels:    Input feature channels.
        out_channels:   Output feature channels.
        stride:         Stride of the convolution (2 = halve spatial size, 1 = keep size).
        use_batchnorm:  Whether to apply BatchNorm (False for the first block).
    """

    def __init__(self, in_channels, out_channels, stride=2, use_batchnorm=True):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=4, stride=stride, padding=1,
                bias=not use_batchnorm
            )
        ]

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(out_channels))

        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

### 2.2 Full PatchGAN Discriminator

**Channel progression** (default: `ndf=64`, `n_layers=3`):

| Layer | stride | Channels |
|-------|--------|----------|
| Block 1 | 2 | 6 → 64 (no BN) |
| Block 2 | 2 | 64 → 128 |
| Block 3 | 2 | 128 → 256 |
| Block 4 | 1 | 256 → 512 (penultimate) |
| Final Conv | 1 | 512 → 1 |

**Feature maps** are collected at each block for the Feature Matching loss (used when training the generator).

In [ ]:
class PatchGANDiscriminator(nn.Module):
    """
    PatchGAN Discriminator for conditional image synthesis.

    Input: concatenation of [semantic_label_map, image] along the channel axis.
    Output:
        - A 2D map of real/fake logits (one score per 70×70 patch for n_layers=3)
        - Intermediate feature maps (for Feature Matching loss)

    Args:
        input_nc:  Channels in the label map.
        output_nc: Channels in the image (real or generated).
        ndf:       Base number of discriminator filters.
        n_layers:  Number of intermediate conv blocks (controls receptive field size).

    Returns (from forward):
        features:   List of intermediate feature tensors (for Feature Matching).
        prediction: Score map [B, 1, H', W'] — higher = more real.
    """

    def __init__(self, input_nc=3, output_nc=3, ndf=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers

        # The discriminator sees label + image concatenated → total input channels
        total_input_nc = input_nc + output_nc  # e.g. 3 + 3 = 6

        self.blocks = nn.ModuleList()

        # ── Block 1: no BatchNorm (raw input shouldn't be normalized) ──
        self.blocks.append(
            DiscriminatorBlock(total_input_nc, ndf, stride=2, use_batchnorm=False)
        )

        # ── Intermediate blocks: stride=2, increasing channel count ────
        in_ch = ndf
        for i in range(1, n_layers):
            out_ch = ndf * min(2 ** i, 8)  # 128, 256, 512 (capped)
            self.blocks.append(
                DiscriminatorBlock(in_ch, out_ch, stride=2, use_batchnorm=True)
            )
            in_ch = out_ch

        # ── Penultimate block: stride=1 (preserves spatial size) ───────
        # This extra block expands the receptive field without halving resolution
        out_ch = ndf * min(2 ** n_layers, 8)
        self.blocks.append(
            DiscriminatorBlock(in_ch, out_ch, stride=1, use_batchnorm=True)
        )
        in_ch = out_ch

        # ── Final layer: output 1-channel logit map ─────────────────────
        # No activation here — loss functions apply sigmoid/softplus internally
        self.final = nn.Conv2d(in_ch, 1, kernel_size=4, stride=1, padding=1)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        """Same N(0, 0.02) initialization as the generator."""
        classname = m.__class__.__name__
        if classname.find("Conv") != -1:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif classname.find("BatchNorm") != -1:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

    def forward(self, label_map, image):
        """
        Args:
            label_map: Semantic label tensor [B, input_nc,  H, W]
            image:     Real or generated image [B, output_nc, H, W]

        Returns:
            features:   List of intermediate feature maps (all blocks except final).
            prediction: Real/fake logit map [B, 1, H', W']
        """
        # Condition on the label: concatenate label + image along channels
        x = torch.cat([label_map, image], dim=1)  # [B, input_nc+output_nc, H, W]

        # Run through all conv blocks and collect feature maps
        features = []
        for block in self.blocks:
            x = block(x)
            features.append(x)  # Used later for Feature Matching loss

        # Final 1-channel prediction (no sigmoid — handled by loss)
        prediction = self.final(x)

        return features, prediction

### ✅ Test the Discriminator

In [ ]:
disc = PatchGANDiscriminator(input_nc=3, output_nc=3, ndf=64, n_layers=3)

label_map = torch.randn(2, 3, 256, 256)
image     = torch.randn(2, 3, 256, 256)

features, pred = disc(label_map, image)

print(f"Label map shape: {label_map.shape}")
print(f"Image shape:     {image.shape}")
print(f"Prediction map:  {pred.shape}  ← one score per 70×70 patch")
print(f"Feature maps:    {[list(f.shape) for f in features]}")

total_params = sum(p.numel() for p in disc.parameters())
print(f"Total parameters: {total_params:,}")
print("✓ Discriminator test passed")

---
# 📉 Part 3 — Loss Functions

The generator is trained with a combination of three losses:

| Loss | Purpose | Weight |
|------|---------|--------|
| **GAN Loss** | Makes generated images look real to the discriminator | 1.0 |
| **L1 Reconstruction** | Pixel-level fidelity to the ground truth | λ = 100 |
| **Feature Matching** | Match discriminator internals of real vs fake | λ = 10 |

The discriminator is only trained with GAN Loss (real vs fake).

### 3.1 GAN Loss

Two modes are supported:

- **`vanilla`** (original GAN): Binary Cross-Entropy with logits  
  `BCEWithLogits(D(fake), 1)` for G;  `BCEWithLogits(D(real), 1)` and `BCEWithLogits(D(fake), 0)` for D

- **`lsgan`** (Least Squares GAN): MSE loss — smoother gradients, more stable training  
  `MSE(D(fake), 1)` for G;  `MSE(D(real), 1) + MSE(D(fake), 0)` for D

In [ ]:
class GANLoss(nn.Module):
    """
    Adversarial loss for GAN training.

    Supports:
        'vanilla' — Binary Cross-Entropy with logits (original GAN formulation)
        'lsgan'   — Least Squares GAN (MSE), which provides smoother gradients
                    and avoids vanishing gradients that vanilla GAN can suffer from

    Args:
        mode:       'vanilla' or 'lsgan'
        real_label: Target value for real samples (default 1.0)
        fake_label: Target value for fake samples (default 0.0)

    Usage:
        loss_fn = GANLoss(mode='lsgan')
        # Training D (real):
        loss_D_real = loss_fn(disc_real_pred, is_real=True)
        # Training D (fake):
        loss_D_fake = loss_fn(disc_fake_pred, is_real=False)
        # Training G:
        loss_G = loss_fn(disc_fake_pred, is_real=True)  # G wants D to think fake=real
    """

    def __init__(self, mode="vanilla", real_label=1.0, fake_label=0.0):
        super().__init__()
        assert mode in ("vanilla", "lsgan"), f"Unknown GAN loss mode: {mode}"
        self.mode = mode

        # Store as buffers so they automatically move to the right device (CPU/GPU)
        # without being treated as learnable parameters
        self.register_buffer("real_val", torch.tensor(real_label))
        self.register_buffer("fake_val", torch.tensor(fake_label))

    def _make_target(self, prediction, is_real):
        """Create a target tensor of the same shape as the discriminator output."""
        value = self.real_val if is_real else self.fake_val
        return value.expand_as(prediction)  # Broadcast scalar to [B, 1, H', W']

    def forward(self, prediction, is_real):
        """
        Args:
            prediction: Discriminator output logits [B, 1, H', W']
            is_real:    True  → compute loss against real_label targets
                        False → compute loss against fake_label targets

        Returns:
            Scalar loss value.
        """
        target = self._make_target(prediction, is_real)

        if self.mode == "vanilla":
            # BCEWithLogits applies sigmoid internally → numerically stable
            loss = F.binary_cross_entropy_with_logits(prediction, target)
        else:  # lsgan
            # LSGAN: penalizes predictions that are far from the target value
            loss = F.mse_loss(prediction, target)

        return loss

### 3.2 L1 Reconstruction Loss

Without L1, a GAN can produce realistic-looking images that are structurally wrong (e.g., completely different layout from the label map). L1 anchors the output to the ground truth at the pixel level.

$$\mathcal{L}_{L1} = \lambda_{L1} \cdot \mathbb{E}\left[ \|y - G(x)\|_1 \right]$$

The weight `λ=100` is the standard pix2pix setting — it strongly encourages fidelity.

In [ ]:
class L1ReconstructionLoss(nn.Module):
    """
    Pixel-wise L1 reconstruction loss.

    The GAN loss alone gives the generator freedom to produce any realistic-
    looking image — not necessarily the correct one. L1 loss constrains the
    output to remain structurally close to the ground truth.

    L1 is preferred over L2 here because:
      - L2 tends to produce blurry outputs (it averages over plausible modes)
      - L1 is less sensitive to outliers and preserves sharper edges

    Args:
        weight: Scaling factor λ_L1 (default 100.0, as in pix2pix paper)

    Returns:
        Weighted mean L1 distance between generated and target images.
    """

    def __init__(self, weight=100.0):
        super().__init__()
        self.weight = weight

    def forward(self, generated, target):
        """
        Args:
            generated: Generator output [B, C, H, W]
            target:    Ground truth image [B, C, H, W]

        Returns:
            Scalar: λ_L1 × mean(|generated - target|)
        """
        # Mean absolute error over all pixels, channels, and batch items
        loss = torch.mean(torch.abs(generated - target))
        return self.weight * loss

### 3.3 Feature Matching Loss

The idea: if the generated image is truly realistic, the discriminator's internal activations (feature maps) when processing the fake image should look similar to when it processes the real image.

$$\mathcal{L}_{FM} = \lambda_{FM} \cdot \frac{1}{N} \sum_{i=1}^{N} \left\| D_i(x, y_{real}) - D_i(x, G(x)) \right\|_1$$

where $D_i$ is the $i$-th intermediate layer of the discriminator.

**Important:** The real features are `detached` — we don't want gradients flowing back through the discriminator when training the generator with this loss.

In [ ]:
class FeatureMatchingLoss(nn.Module):
    """
    Feature Matching Loss.

    Encourages the generator to match the discriminator's internal feature
    statistics when comparing real vs generated images. This improves training
    stability by giving the generator a richer learning signal beyond just
    the final real/fake decision.

    How it works:
      1. Run the discriminator on the real image → collect feature maps {D_i(real)}
      2. Run the discriminator on the fake image → collect feature maps {D_i(fake)}
      3. Minimize the L1 distance between each pair of feature maps

    Args:
        weight: Scaling factor λ_FM (default 10.0)

    Returns:
        Weighted average L1 distance across all discriminator layers.
    """

    def __init__(self, weight=10.0):
        super().__init__()
        self.weight = weight

    def forward(self, features_real, features_fake):
        """
        Args:
            features_real: List of feature tensors from D(label, real_image).
                           Typically has the same length as the number of D blocks.
            features_fake: List of feature tensors from D(label, fake_image).

        Returns:
            Scalar: λ_FM × mean L1 distance across all feature layers.
        """
        assert len(features_real) == len(features_fake), \
            "Real and fake feature lists must have the same number of layers"

        loss = 0.0
        for feat_real, feat_fake in zip(features_real, features_fake):
            # detach() is CRITICAL here:
            # We don't want gradients to flow back through the discriminator
            # when computing the generator's feature matching loss.
            # The discriminator is updated separately via its own optimizer.
            loss += torch.mean(torch.abs(feat_real.detach() - feat_fake))

        # Average over layers so the magnitude doesn't scale with network depth
        loss = loss / len(features_real)
        return self.weight * loss

### ✅ Test All Losses

In [ ]:
# ── GAN Loss ─────────────────────────────────────────────────────────────
print("── GAN Loss ──")
for mode in ["vanilla", "lsgan"]:
    gan_loss = GANLoss(mode=mode)
    pred = torch.randn(2, 1, 30, 30)  # Simulated discriminator output
    print(f"  [{mode}] real loss: {gan_loss(pred, is_real=True).item():.4f}  "
          f"fake loss: {gan_loss(pred, is_real=False).item():.4f}")

# ── L1 Reconstruction Loss ───────────────────────────────────────────────
print("\n── L1 Reconstruction Loss ──")
l1_loss = L1ReconstructionLoss(weight=100.0)
gen_img = torch.randn(2, 3, 256, 256)
real_img = torch.randn(2, 3, 256, 256)
loss_l1 = l1_loss(gen_img, real_img)
print(f"  L1 loss (random inputs, weight=100): {loss_l1.item():.2f}")
# When identical inputs: loss should be 0
print(f"  L1 loss (identical inputs):          {l1_loss(gen_img, gen_img).item():.4f}")

# ── Feature Matching Loss ─────────────────────────────────────────────────
print("\n── Feature Matching Loss ──")
fm_loss = FeatureMatchingLoss(weight=10.0)
# Simulate 4 feature map pairs from the discriminator
feats_real = [torch.randn(2, 64, 128, 128), torch.randn(2, 128, 64, 64),
              torch.randn(2, 256, 32, 32),  torch.randn(2, 512, 16, 16)]
feats_fake = [torch.randn(2, 64, 128, 128), torch.randn(2, 128, 64, 64),
              torch.randn(2, 256, 32, 32),  torch.randn(2, 512, 16, 16)]
loss_fm = fm_loss(feats_real, feats_fake)
print(f"  FM loss (random features, weight=10): {loss_fm.item():.4f}")

print("\n✓ All loss tests passed")

---
# 🔗 Part 4 — End-to-End Integration Check

Let's wire the generator, discriminator, and all three losses together in a single forward pass — exactly as they would run during one training step.

In [ ]:
# ── Instantiate all components ────────────────────────────────────────────
generator     = UNetGenerator(input_nc=3, output_nc=3, ngf=64, depth=8)
discriminator = PatchGANDiscriminator(input_nc=3, output_nc=3, ndf=64, n_layers=3)

gan_loss = GANLoss(mode="lsgan")
l1_loss  = L1ReconstructionLoss(weight=100.0)
fm_loss  = FeatureMatchingLoss(weight=10.0)

# ── Dummy batch ───────────────────────────────────────────────────────────
label_map  = torch.randn(1, 3, 256, 256)   # Semantic label map (input)
real_image = torch.randn(1, 3, 256, 256)   # Ground truth image

# ── Generator forward pass ────────────────────────────────────────────────
fake_image = generator(label_map)           # G(label) → synthesized image

# ── Discriminator forward passes ─────────────────────────────────────────
feats_real, pred_real = discriminator(label_map, real_image)  # D on real
feats_fake, pred_fake = discriminator(label_map, fake_image)  # D on fake

# ── Discriminator losses ──────────────────────────────────────────────────
# D wants: real → 1, fake → 0
loss_D_real = gan_loss(pred_real, is_real=True)
loss_D_fake = gan_loss(pred_fake, is_real=False)
loss_D = (loss_D_real + loss_D_fake) * 0.5   # Average as is standard

# ── Generator losses ──────────────────────────────────────────────────────
# G wants: discriminator to think fake → real
loss_G_adv = gan_loss(pred_fake, is_real=True)
loss_G_l1  = l1_loss(fake_image, real_image)
loss_G_fm  = fm_loss(feats_real, feats_fake)

# Total generator loss (weighted combination)
loss_G = loss_G_adv + loss_G_l1 + loss_G_fm

# ── Print summary ─────────────────────────────────────────────────────────
print("=== Discriminator ===")
print(f"  D(real) loss : {loss_D_real.item():.4f}")
print(f"  D(fake) loss : {loss_D_fake.item():.4f}")
print(f"  Total D loss : {loss_D.item():.4f}")

print("\n=== Generator ===")
print(f"  Adversarial   (GAN)  : {loss_G_adv.item():.4f}")
print(f"  Reconstruction (L1)  : {loss_G_l1.item():.4f}")
print(f"  Feature Matching     : {loss_G_fm.item():.4f}")
print(f"  Total G loss         : {loss_G.item():.4f}")

print("\n=== Shapes ===")
print(f"  Input label map : {label_map.shape}")
print(f"  Fake image      : {fake_image.shape}")
print(f"  D output (pred) : {pred_fake.shape}")

print("\n✓ End-to-end integration check passed")

---
# 📋 Summary

| Component | Key Design Choices |
|-----------|-------------------|
| **UNetGenerator** | Symmetric encoder-decoder, skip connections at every level, Tanh output |
| **PatchGANDiscriminator** | Classifies 70×70 patches, conditional on label map, returns feature maps |
| **GANLoss** | Supports vanilla BCE or LSGAN (MSE); target tensors auto-shaped |
| **L1ReconstructionLoss** | Pixel fidelity, λ=100 anchors output close to ground truth |
| **FeatureMatchingLoss** | Matches discriminator internals of real vs fake, λ=10 |

### Next Steps
- Add a **training loop** with separate G and D optimizers (Adam, lr=2e-4, β₁=0.5)
- Add a **dataloader** for your semantic label / image pairs
- Add **validation** with FID or LPIPS metrics
- Consider **SPADE normalization** for better style control on diverse label maps